# Getting Your Instrument Data Into Python (Without Losing Your Mind)
### 2.4 — Reading Real Instrument Files (CSV, TXT, and the Messy Ones)

Every analytical method ends the same way: an instrument writes a file, and you
have to get that file *into Python* before you can do anything useful with it.
This is the first wall most scientists hit — and it is almost never a clean
`pd.read_csv("data.csv")`.

Vendor software exports for *its own* convenience, not for pandas. So you get
metadata headers, semicolons instead of commas, commas instead of decimal
points, footer junk, comment lines, and missing values written five different
ways. None of it is hard once you've seen it. The trick is a **repeatable
habit**, not a pile of one-off recipes.

**By the end of this notebook you will be able to:**

1. **Look at a raw file *before* parsing it** — the single habit that saves the
   most time.
2. **Skip metadata headers and footer junk** with `skiprows`, `skipfooter`, and
   `comment`.
3. **Handle non-standard delimiters and decimal separators** (`sep`, `decimal`,
   whitespace).
4. **Clean, type-check, and sanity-check** a spectrum once it's in a DataFrame.
5. **Wrap the whole thing in one reusable loader** you can drop into every later
   notebook.

> **Reproducibility:** we generate the messy files ourselves from a fixed random
> seed, so the bytes on your disk are identical to the bytes on ours. Run the
> notebook top to bottom and every number below will match.

In [ ]:
# Standard scientific Python stack -- nothing exotic.
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Our project's data simulator. It returns a `Dataset` with a shared axis (`x`),
# a 2D data matrix (`X`), labels/units, and a `meta` table of ground truth.
#
# This import works from anywhere once you've installed the package once, from
# the repo root:   pip install -e .
# (That puts `simulated_data` on your Python path regardless of which folder this
# notebook lives in.)
from simulated_data import uvvis

# A folder to hold the messy export files we are about to manufacture.
# Jupyter and nbconvert run a notebook with ITS OWN folder as the working
# directory, so this relative path lands right next to this notebook. It's
# regenerated on every run and git-ignored -- safe to delete any time.
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

pd.set_option("display.max_rows", 8)  # keep DataFrame previews short
print("pandas", pd.__version__, "| files will be written to:", EXPORTS.resolve())

## 1. Why instrument files fight back

Four obstacles account for the vast majority of "my file won't load" moments:

| Obstacle | What it looks like | Why it happens |
|---|---|---|
| **Metadata headers** | 5–30 lines of instrument settings before the data | The vendor stamps every export with method, operator, date, serial number |
| **Odd delimiters** | `;` instead of `,`, or tabs, or runs of spaces | Locale settings, or "we used tabs so commas in text don't break it" |
| **Extra rows** | footer lines like `*** End of Data ***`, blank lines | Software writes a trailer, or a UI control character |
| **Inconsistent formatting** | `1,5` for 1.5, `N/A` for missing, `#` comments mid-file | Locale decimal commas, hand-edited files, logging quirks |

We'll **manufacture each of these on purpose**. Once you've built the mess, it's
never mysterious when you meet it in the wild.

## 2. Start from a clean spectrum (so we know the right answer)

First we simulate one believable UV-Vis absorbance spectrum. Because it's
simulated, we know the *true* numbers — so after every rescue below we can
confirm we recovered **exactly** the data we started with. Real files can never
offer that.

In [ ]:
# One reproducible UV-Vis spectrum: a curved baseline + a little noise,
# over the visible range (400-800 nm). `seed` makes it identical every run.
ds = uvvis.simulate(seed=7, baseline=0.15)

wavelength = ds.x            # 1D wavelength axis, shape (n_points,)
absorbance = ds.single()     # the lone spectrum as a 1D array

# The TIDY two-column table every messy file below should reduce to.
clean = pd.DataFrame({"Wavelength_nm": wavelength, "Absorbance": absorbance})

print("axis:", ds.x_label, f"({ds.x_unit})", "| signal:", ds.y_label)
print("points:", len(clean), "| wavelength span:",
      f"{wavelength.min():.0f}-{wavelength.max():.0f} nm")
clean.head()

That `clean` table — wavelength in one column, absorbance in the next, correct
dtypes, sorted by wavelength — is our **target shape**. Keep it in mind: every
parsing problem below is just a detour back to this same table.

## 3. Manufacturing four realistic messy exports

We now write four files to `exports/`, each reproducing a real vendor quirk.
We use plain Python `write_text` so you can see **exactly** what lands on disk —
no pandas magic hiding the mess.

In [ ]:
# Helper: format the spectrum as "value<sep>value" rows, with a chosen
# decimal mark and number formatting. This lets us reproduce locale quirks.
def rows(sep=",", decimal=".", fmt="{:.1f}", yfmt="{:.4f}"):
    lines = []
    for wl, ab in zip(wavelength, absorbance):
        wl_s = fmt.format(wl).replace(".", decimal)
        ab_s = yfmt.format(ab).replace(".", decimal)
        lines.append(f"{wl_s}{sep}{ab_s}")
    return lines

print("example data row (comma CSV):", rows()[0])

### File A — vendor CSV with a metadata header
The classic. A block of instrument settings, a blank line, then the real column
header and comma-separated data.

In [ ]:
file_a = EXPORTS / "uvvis_vendor_metadata.csv"

header_a = [
    "Instrument: SpectraMax UV-2600",
    "Serial Number: UV26-00471",
    "Operator: C. Pulliam",
    "Acquisition Date: 2026-06-12 09:14:03",
    "Method: Absorbance_Scan_400-800nm",
    "Integration Time (ms): 100",
    "",                                   # blank separator line
    "Wavelength (nm),Absorbance",         # the REAL column header
]
file_a.write_text("\n".join(header_a + rows(sep=",")) + "\n")
print(f"wrote {file_a}  ({file_a.stat().st_size} bytes)")

### File B — European export (semicolon delimiter, comma decimals)
The silent killer. Many instruments in locale settings write `450,0;0,8231` —
semicolon between columns, **comma as the decimal point**. Load it naively and
your numbers quietly turn into text.

In [ ]:
file_b = EXPORTS / "uvvis_european.csv"

header_b = ["Wellenlaenge (nm);Absorption"]   # German-style header, semicolon
file_b.write_text(
    "\n".join(header_b + rows(sep=";", decimal=",")) + "\n"
)
print(f"wrote {file_b}  ({file_b.stat().st_size} bytes)")

### File C — tab-delimited TXT with a footer
A `.txt` export with a metadata header, **tab** delimiters, and trailing footer
rows (`*** End of Data ***`, a checksum) plus a blank line at the end.

In [ ]:
file_c = EXPORTS / "uvvis_tabbed.txt"

header_c = [
    "# Raman/UV Export Utility v3.2",
    "# Sample: standard_001",
    "Wavelength\tAbsorbance",          # tab between the two column names
]
footer_c = ["*** End of Data ***", "Checksum: 0x4F2A", ""]  # trailing junk
file_c.write_text(
    "\n".join(header_c + rows(sep="\t") + footer_c) + "\n"
)
print(f"wrote {file_c}  ({file_c.stat().st_size} bytes)")

### File D — inconsistent formatting (whitespace, comments, missing values)
The catch-all. Variable runs of spaces as the delimiter, `#` comment lines
scattered through the file, and a couple of missing readings written as `N/A`.

In [ ]:
file_d = EXPORTS / "uvvis_messy.dat"

# Build whitespace-delimited rows with irregular spacing, inject comments,
# and blank out two absorbance values as "N/A" (a detector dropout).
lines_d = ["# Logger dump -- columns: wavelength  absorbance",
           "# units: nm, AU"]
for i, (wl, ab) in enumerate(zip(wavelength, absorbance)):
    if i in (50, 150):                       # two simulated missing readings
        ab_s = "N/A"
    else:
        ab_s = f"{ab:.4f}"
    pad = "   " if i % 2 == 0 else "  "       # deliberately uneven spacing
    lines_d.append(f"{wl:.1f}{pad}{ab_s}")
    if i == 100:
        lines_d.append("# --- detector autogain event ---")  # comment mid-data

file_d.write_text("\n".join(lines_d) + "\n")
print(f"wrote {file_d}  ({file_d.stat().st_size} bytes)")

## 4. The habit that saves hours: *look before you load*

Before writing a single `read_csv`, **print the first few raw lines**. Ten
seconds of looking tells you where the data starts, what the delimiter is, and
how numbers are formatted — which is everything you need to parse correctly.

In [ ]:
def peek(path, n=12):
    "Print the first n raw lines of a file, with line numbers."
    text = Path(path).read_text()
    for i, line in enumerate(text.splitlines()[:n]):
        # repr() makes tabs (\t) and hidden characters visible.
        print(f"{i:>2}: {line!r}")

print("===== File A =====")
peek(file_a)

Reading File A by eye:

- Lines **0–5** are metadata (`Instrument:`, `Serial Number:`, ...).
- Line **6** is blank.
- Line **7** is the real header: `'Wavelength (nm),Absorbance'`.
- Line **8** onward is comma-separated data.

So: the data starts at line 7, and the delimiter is a comma. That's the whole
puzzle solved before we've touched pandas. Let's peek at the other three.

In [ ]:
for label, path in [("File B", file_b), ("File C", file_c), ("File D", file_d)]:
    print(f"===== {label} =====")
    peek(path, n=6)
    print()

Notice what the `repr()` view reveals:

- **File B** uses `;` between columns and `,` inside numbers (`'450,0;0,8231'`).
- **File C** shows `\t` (a tab) between `Wavelength` and `Absorbance`, and `#`
  comment lines up top.
- **File D** has `#` comments, irregular spaces, and an `N/A` lurking further
  down.

Now each fix is obvious. Let's do them one at a time.

## 5. Rescue, file by file

For each file: a naive attempt that fails or *looks* wrong, then the fix, then a
check against the ground truth. The pattern is always the same.

### 5A. Metadata header → `skiprows`

The data starts at line 7 (the real header). Tell pandas to skip the first 7
lines. Equivalent and more robust options are mentioned after.

In [ ]:
# Naive attempt: pandas treats the metadata as data and chokes / mislabels.
naive_a = pd.read_csv(file_a, nrows=3)
print("Naive read -- pandas thinks the columns are:")
print(list(naive_a.columns), "\n")

# Fix: skip the 7 metadata+blank lines so the real header is row 0.
df_a = pd.read_csv(file_a, skiprows=7)
print("Fixed read:")
print(df_a.head(3))
print("\ncolumns:", list(df_a.columns), "| dtypes:\n", df_a.dtypes)

**More robust alternatives** to a hard-coded `skiprows=7`:

- If the metadata lines share a prefix you could pass `comment=...`, but here
  they don't, so counting is fine.
- The bullet-proof approach — *find the header automatically* — is exactly what
  our reusable loader in Section 7 does. Hard-coding `skiprows` is fine for a
  one-off; auto-detection is better for a function you'll reuse.

### 5B. European decimals → `sep=';', decimal=','`

This is the dangerous one, because the naive read **doesn't raise an error** —
it just gives you the wrong data. Watch the dtype.

In [ ]:
# Naive: we get the delimiter right (sep=';') but FORGET the decimal comma.
# The columns split correctly, but '400,0' isn't a valid number, so pandas
# keeps them as text ('object'). No error is raised -- just silently broken.
naive_b = pd.read_csv(file_b, sep=";")
print("Naive read of File B (sep=';' but no decimal=','):")
print("  columns:", list(naive_b.columns))
print("  dtypes :", naive_b.dtypes.to_dict(), "<- 'object' means text, not numbers!")
print("  sample value:", repr(naive_b.iloc[0, 0]), "<- a STRING, not 400.0\n")

# Fix: declare BOTH the column separator and the decimal mark.
df_b = pd.read_csv(file_b, sep=";", decimal=",")
print("Fixed read of File B:")
print(df_b.head(3))
print("\n  dtypes:", df_b.dtypes.to_dict(), "<- float64, as it should be")

> **The lesson that bites real labs:** a misread decimal comma doesn't crash —
> it sails downstream into your calibration and quietly biases every result.
> Always check `df.dtypes` after loading. If a numeric column is `object`, your
> numbers are still text.

### 5C. Tab + footer → `sep='\t', skipfooter=..., comment='#'`

Tab delimiter, two `#` comment lines on top, and three footer lines at the
bottom. `comment='#'` drops the comments; `skipfooter` drops the trailer.
`skipfooter` requires the Python parser engine.

In [ ]:
df_c = pd.read_csv(
    file_c,
    sep="\t",            # tab-delimited
    comment="#",         # drop the "# ..." lines at the top
    skipfooter=3,        # drop "*** End of Data ***", checksum, blank line
    engine="python",     # skipfooter is only supported by the python engine
)
print(df_c.head(3))
print("...")
print(df_c.tail(3))
print("\nrows parsed:", len(df_c), "| dtypes:", df_c.dtypes.to_dict())

The footer is gone, the comments are gone, and we're left with clean numeric
columns. `engine="python"` is slower than the default C engine but is the price
of `skipfooter` — fine for the file sizes spectroscopy produces.

### 5D. Inconsistent whitespace → `sep=r'\s+', comment='#', na_values=['N/A']`

Variable spaces as the delimiter (so we match *one-or-more* whitespace with the
regex `\s+`), `#` comments to drop, and `N/A` to recognize as missing.

In [ ]:
df_d = pd.read_csv(
    file_d,
    sep=r"\s+",                 # one or more whitespace chars = the delimiter
    comment="#",                # drop "# ..." lines wherever they appear
    names=["Wavelength", "Absorbance"],  # the file has no real header row
    na_values=["N/A"],          # turn the "N/A" strings into real NaN
    engine="python",
)
print(df_d.head(3))
print("\nmissing values per column:\n", df_d.isna().sum())
print("\nrows flagged missing:")
print(df_d[df_d["Absorbance"].isna()])

Because we declared `na_values=["N/A"]`, pandas turned those two dropouts into
real `NaN` instead of poisoning the whole column to text. Now we can *decide*
what to do with them (drop, interpolate, flag) — covered next.

## 6. Clean and inspect once it's in

Parsing is only half the job. Before trusting the data, run a short checklist
that turns a parsed table into *trustworthy* data. We'll use File A's result and
verify it against the ground truth.

In [ ]:
def tidy(df, wl_col, ab_col):
    "Standardize column names, coerce to numeric, sort by wavelength, reset index."
    out = df.rename(columns={wl_col: "Wavelength_nm", ab_col: "Absorbance"})
    # to_numeric with errors='coerce' makes any stray text become NaN (safe).
    out["Wavelength_nm"] = pd.to_numeric(out["Wavelength_nm"], errors="coerce")
    out["Absorbance"] = pd.to_numeric(out["Absorbance"], errors="coerce")
    out = out.dropna(subset=["Wavelength_nm"])          # need a valid axis
    out = out.sort_values("Wavelength_nm").reset_index(drop=True)
    return out[["Wavelength_nm", "Absorbance"]]


clean_a = tidy(df_a, "Wavelength (nm)", "Absorbance")
clean_a.head(3)

In [ ]:
# A quick sanity-check function: are the basic physical facts plausible?
def sanity_check(df):
    wl = df["Wavelength_nm"].to_numpy()
    ab = df["Absorbance"].to_numpy()
    print("rows                :", len(df))
    print("wavelength range    : {:.1f} - {:.1f} nm".format(wl.min(), wl.max()))
    print("axis monotonic up?  :", bool(np.all(np.diff(wl) > 0)))
    print("absorbance range    : {:.3f} - {:.3f} AU".format(np.nanmin(ab), np.nanmax(ab)))
    print("any NaN absorbance? :", bool(df["Absorbance"].isna().any()))

sanity_check(clean_a)

In [ ]:
# Ground-truth check: did we recover the EXACT spectrum we started with?
# (File A was written at full precision, so the numbers should match closely.)
recovered = clean_a["Absorbance"].to_numpy()
truth = clean["Absorbance"].to_numpy()

print("same number of points:", recovered.shape == truth.shape)
print("max abs difference   :", np.max(np.abs(recovered - truth)))
print("np.allclose to truth :", np.allclose(recovered, truth, atol=1e-4))

`np.allclose` confirms the bytes-on-disk round-tripped back to the spectrum we
simulated (within the 4-decimal precision we exported at). *That* is what "I
loaded it correctly" actually means — not just "it didn't error."

## 7. One loader to rule them all

Doing this by hand four times taught the moves. Now bottle them into a single
function that **diagnoses the file for you**: it sniffs the delimiter, finds the
first real data row (skipping any metadata header), and returns the tidy table.
This is the helper you'll import into every later notebook.

In [ ]:
import csv


def load_spectrum(path, na_values=("N/A", "NA", "")):
    '''Load a messy instrument export into a tidy two-column DataFrame.

    Strategy (the same habit, automated):
      1. Read the raw lines.
      2. Skip comment ('#') and blank lines.
      3. Find the first line that looks like NUMERIC data -> data starts there.
      4. Sniff the delimiter from that line (',', ';', tab, or whitespace).
      5. Detect a comma decimal mark if there are no '.' but there are ','.
      6. Parse, coerce to numbers, sort, and return [Wavelength_nm, Absorbance].
    '''
    raw = Path(path).read_text().splitlines()

    # Keep only content lines (drop comments and blanks) but remember order.
    content = [ln for ln in raw if ln.strip() and not ln.lstrip().startswith("#")]

    def looks_numeric(line):
        # Replace likely delimiters with spaces, comma-decimal with dot, and
        # check that the resulting tokens are all numbers.
        tokens = line.replace(";", " ").replace(",", " ").replace("\t", " ").split()
        if len(tokens) < 2:
            return False
        try:
            [float(t) for t in tokens]
            return True
        except ValueError:
            return False

    # First numeric line = start of data (everything above it is header junk).
    start = next((i for i, ln in enumerate(content) if looks_numeric(ln)), None)
    if start is None:
        raise ValueError(f"No numeric data rows found in {path}")
    data_lines = content[start:]

    # Sniff the delimiter from the first data line.
    sample = data_lines[0]
    if "\t" in sample:
        sep = "\t"
    elif ";" in sample:
        sep = ";"
    elif "," in sample and sample.count(",") > sample.count(" "):
        sep = ","
    else:
        sep = r"\s+"   # fall back to whitespace

    # Comma-decimal detection: commas present but no dots -> decimal=','
    decimal = "," if ("," in sample and "." not in sample and sep != ",") else "."

    df = pd.read_csv(
        io.StringIO("\n".join(data_lines)),
        sep=sep,
        decimal=decimal,
        header=None,
        names=["Wavelength_nm", "Absorbance"],
        na_values=list(na_values),
        engine="python",
    )
    df["Wavelength_nm"] = pd.to_numeric(df["Wavelength_nm"], errors="coerce")
    df["Absorbance"] = pd.to_numeric(df["Absorbance"], errors="coerce")
    df = df.dropna(subset=["Wavelength_nm"]).sort_values("Wavelength_nm")
    return df.reset_index(drop=True)

In [ ]:
# One function, every file. The whole point: the caller doesn't care which
# quirk each file had.
loaded = {}
for path in [file_a, file_b, file_c, file_d]:
    df = load_spectrum(path)
    loaded[path.name] = df
    print(f"{path.name:28s} -> {len(df):3d} rows, "
          f"{df['Wavelength_nm'].min():.0f}-{df['Wavelength_nm'].max():.0f} nm, "
          f"dtypes {df['Absorbance'].dtype}")

In [ ]:
# Confirm every file recovered the same underlying spectrum. All files store
# rows in ascending-wavelength order with the same 400 points as the truth, so
# we compare absorbance position-by-position. File D has two intentional N/A
# dropouts, so we mask out NaNs before comparing.
truth_ab = clean["Absorbance"].to_numpy()      # full-precision ground truth
for name, df in loaded.items():
    rec = df["Absorbance"].to_numpy()
    if rec.shape != truth_ab.shape:
        print(f"{name:28s} row-count mismatch ({rec.shape} vs {truth_ab.shape})")
        continue
    mask = ~np.isnan(rec)                       # skip File D's two dropouts
    ok = np.allclose(rec[mask], truth_ab[mask], atol=1e-3)
    print(f"{name:28s} matches ground truth: {ok}  "
          f"(compared {mask.sum()} of {len(rec)} points)")

## 8. Plot the rescued data

The payoff: bytes-on-disk became chemistry-in-a-figure. We overlay all four
recovered spectra — they sit exactly on top of each other, which is the visual
proof that all four parses agreed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for name, df in loaded.items():
    ax.plot(df["Wavelength_nm"], df["Absorbance"], label=name, alpha=0.8)

# Labels/units come straight from the simulated Dataset, so they're always right.
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("Four messy exports, one recovered spectrum")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

All four lines coincide — every file, despite its own quirk, reduced to the same
underlying UV-Vis spectrum. Two clean absorbance bands (~450 nm and ~600 nm) on
a gently curved baseline, exactly what we simulated.

## 9. What this means scientifically

Reading the file correctly **is part of the measurement.** A misparsed decimal
separator or an un-skipped footer rarely throws an error — it silently corrupts
the numbers, and that corruption flows straight into your baseline correction,
peak integration, calibration curve, and any model you build downstream.

The defensive habits worth keeping:

- **Look at the raw file first.** Ten seconds of reading beats an hour of
  debugging a wrong parse.
- **Check `df.dtypes` after every load.** A numeric column showing `object` means
  your numbers are still text — the most common silent failure.
- **Verify the axis is monotonic and in the expected range.** Spectrometers
  occasionally export descending or doubled axes.
- **Decide what missing values mean** before you average over them.

Do this once, wrap it in a function, and the rest of your analysis stands on
solid ground.

## Key Takeaways

- **Reading the file correctly is part of the measurement.** A misparsed decimal
  mark or an un-skipped footer rarely throws an error — it silently corrupts the
  numbers, and that corruption flows into every downstream step.
- **One repeatable loop works on any export:** *Diagnose* (peek at the raw
  lines) → *Load* (choose `sep` / `decimal` / `skiprows` / `comment` from what
  you saw) → *Clean* (rename, `to_numeric`, sort, handle NaNs) → *Inspect*
  (dtypes, monotonic axis, physical ranges) → *Plot* (confirm visually).
- **Wrap it in a function.** Once `load_spectrum()` absorbs the per-file quirks,
  the rest of your analysis never has to think about them again.

## Practical Checklist

- [ ] Print the first ~12 raw lines **before** parsing.
- [ ] Pick `sep` / `decimal` / `skiprows` / `comment` from the raw view, not by
  guessing.
- [ ] After loading, check `df.dtypes` — a numeric column showing `object` means
  your numbers are still text.
- [ ] Confirm the x-axis is **monotonic** and within the expected physical range.
- [ ] Decide what missing values mean *before* you average over them.
- [ ] Where a reference exists, verify the parse against it (`np.allclose`).

## Common Mistakes

- Numbers loaded as text because of a comma decimal mark — silent, and it breaks
  every later calculation.
- Header or footer rows left in, turning into `NaN`s or shifting columns.
- Assuming an ascending axis; some instruments export descending or doubled axes.
- Averaging or interpolating across `N/A` dropouts without deciding what they
  represent.

## Reporting Guidance

- Record **how** each file was parsed (delimiter, decimal mark, skipped rows)
  alongside the results — it is part of the data's provenance.
- Keep the raw export untouched and do all cleaning in code, so the path from
  bytes-on-disk to numbers is reproducible and auditable.

## Next Lesson

**2.5 — Plotting That Reveals Chemistry.** We now have clean spectra in memory;
next we turn them into figures that answer chemical questions.